<a href="https://colab.research.google.com/github/shuvo4758/PyTorch_Project_2-RNN-Based-Question-System/blob/main/RNN_Based_Question_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loading Dataset

In [193]:
import pandas as pd

df = pd.read_csv('/content/100_Unique_QA_Dataset.csv')

print(df.head(10))

                                          question             answer
0                   What is the capital of France?              Paris
1                  What is the capital of Germany?             Berlin
2               Who wrote 'To Kill a Mockingbird'?         Harper-Lee
3  What is the largest planet in our solar system?            Jupiter
4   What is the boiling point of water in Celsius?                100
5                       Who painted the Mona Lisa?  Leonardo-da-Vinci
6                   What is the square root of 64?                  8
7            What is the chemical symbol for gold?                 Au
8                 Which year did World War II end?               1945
9          What is the longest river in the world?               Nile


# Tokenizing --> spliting words

In [194]:
def tokenize(text):
  text = text.lower()   # converting into lower case
  text = text.replace("'","")   # removing (')
  text = text.replace('?', '')    # removing (?)
  return text.split()   # spliting words

In [195]:
# checking tokenize function working or not
tokenize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

# Vocabulary

In [196]:
vocab = {'<UNK>': 0}   # unknown words vocabulary is set to index 0

In [197]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  merged_tokens = tokenized_question + tokenized_answer

  # adding tokens into vocab using for loop
  for token in merged_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)

In [198]:
# calling build_vocab function
df.apply(build_vocab, axis=1)

# length of the vocab --> total unique words
print('Total unique words:',len(vocab))

Total unique words: 324


# Converting words to numerical indices

In [199]:
def text_to_indices(text, vocab):

  indexed_text = []

  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [200]:
# testing text_to_indices function
text_to_indices('What is campusx', vocab)

[1, 2, 0]

In [201]:
import torch
from torch.utils.data import Dataset, DataLoader

In [202]:
class QADataset(Dataset):
  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):
    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [203]:
dataset = QADataset(df, vocab)

In [204]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [205]:
# for question, answer in dataloader:
#   print(question, answer)

# RNN Architecture

In [206]:
import torch.nn as nn

In [207]:
class SimpleRNN(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))
    return output

In [208]:
learning_rate = 0.001
epochs = 20

In [209]:
model = SimpleRNN(len(vocab))

In [210]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Training loop

In [211]:
for epoch in range(epochs):
  total_loss = 0

  for question, answer in dataloader:
    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss
    loss = loss_fn(output, answer[0])

    # gradients
    loss.backward()

    # update weights
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f'Epoch: {epoch+1}/{epochs} Loss: {total_loss:.4f}')

Epoch: 1/20 Loss: 522.3112
Epoch: 2/20 Loss: 447.0557
Epoch: 3/20 Loss: 368.7381
Epoch: 4/20 Loss: 310.7014
Epoch: 5/20 Loss: 259.7732
Epoch: 6/20 Loss: 211.6570
Epoch: 7/20 Loss: 167.3704
Epoch: 8/20 Loss: 130.6008
Epoch: 9/20 Loss: 99.9792
Epoch: 10/20 Loss: 76.6243
Epoch: 11/20 Loss: 59.6226
Epoch: 12/20 Loss: 46.9311
Epoch: 13/20 Loss: 37.5237
Epoch: 14/20 Loss: 30.5119
Epoch: 15/20 Loss: 25.5095
Epoch: 16/20 Loss: 21.6946
Epoch: 17/20 Loss: 18.3161
Epoch: 18/20 Loss: 15.6502
Epoch: 19/20 Loss: 13.4511
Epoch: 20/20 Loss: 11.8084


# Prediction

In [212]:
def predict(model, question, threshold=0.5):

  # converting question to number
  numerical_question = text_to_indices(question, vocab)

  # converting numerical_question to tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # sending question_tensor to model
  output = model(question_tensor)

  # converting logits to probabilities
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print('I Don\'t know')
  else:
    list(vocab.keys())[index]
    print(list(vocab.keys())[index])

In [213]:
predict(model, 'What is campusx')

I Don't know


In [214]:
predict(model, 'What is the largest planet in our solar system')

jupiter


In [215]:
predict(model, 'What is the boiling point of water in Celsius?')

100


In [216]:
predict(model, 'Mona Lisa?')

leonardo-da-vinci


In [217]:
predict(model, 'capital of Germany?')

berlin
